# Embeddings pré-treinados: Word2Vec, FastText e GloVe

Este notebook é apenas para testar modelos pré-treinados. Ele carrega vetores prontos, compara palavras por similaridade e permite visualizar palavras próximas em 2D.

Observação: os modelos Word2Vec Google News e FastText Wiki News são grandes e podem demorar no Colab. Se quiser um teste rápido, carregue primeiro apenas o GloVe.

In [ ]:
%pip install -q gensim scikit-learn pandas matplotlib

In [ ]:
import gensim.downloader as api
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

## Modelos disponíveis

- `glove-wiki-gigaword-100`: GloVe pré-treinado, menor e mais rápido.
- `word2vec-google-news-300`: Word2Vec pré-treinado, grande.
- `fasttext-wiki-news-subwords-300`: FastText pré-treinado, grande e com suporte a subpalavras.

In [ ]:
MODELOS_PRE_TREINADOS = {
    "glove": "glove-wiki-gigaword-100",
    "word2vec": "word2vec-google-news-300",
    "fasttext": "fasttext-wiki-news-subwords-300",
}

# Para Colab com pouco tempo, use ["glove"].
# Para cumprir a comparação completa, use ["glove", "word2vec", "fasttext"].
modelos_para_carregar = ["glove"]

modelos = {}
for nome, codigo in MODELOS_PRE_TREINADOS.items():
    if nome in modelos_para_carregar:
        print(f"Carregando {nome}: {codigo}")
        modelos[nome] = api.load(codigo)
        print(f"{nome} carregado com {len(modelos[nome].index_to_key):,} palavras.")

In [ ]:
def palavras_similares(modelo, palavra, topn=10):
    if palavra not in modelo:
        return pd.DataFrame({"aviso": [f"A palavra '{palavra}' não está no vocabulário do modelo."]})
    return pd.DataFrame(modelo.most_similar(palavra, topn=topn), columns=["palavra", "similaridade"])


palavra_teste = "language"

for nome, modelo in modelos.items():
    print(f"\nModelo: {nome}")
    display(palavras_similares(modelo, palavra_teste, topn=10))

In [ ]:
def comparar_pares(modelo, pares):
    linhas = []
    for a, b in pares:
        if a in modelo and b in modelo:
            score = modelo.similarity(a, b)
            linhas.append({"palavra_1": a, "palavra_2": b, "similaridade": score})
        else:
            linhas.append({"palavra_1": a, "palavra_2": b, "similaridade": None})
    return pd.DataFrame(linhas)


pares = [
    ("language", "linguistics"),
    ("language", "computer"),
    ("translation", "language"),
    ("text", "corpus"),
]

for nome, modelo in modelos.items():
    print(f"\nModelo: {nome}")
    display(comparar_pares(modelo, pares))

In [ ]:
def plotar_vizinhos(modelo, palavra, topn=20, titulo=None):
    if palavra not in modelo:
        print(f"A palavra '{palavra}' não está no vocabulário.")
        return

    vizinhos = [palavra] + [w for w, _ in modelo.most_similar(palavra, topn=topn)]
    vetores = [modelo[w] for w in vizinhos]
    coords = PCA(n_components=2, random_state=42).fit_transform(vetores)

    plt.figure(figsize=(10, 7))
    plt.scatter(coords[:, 0], coords[:, 1])
    for i, token in enumerate(vizinhos):
        plt.annotate(token, (coords[i, 0], coords[i, 1]))
    plt.title(titulo or f"Vizinhos de '{palavra}'")
    plt.grid(alpha=0.2)
    plt.show()


for nome, modelo in modelos.items():
    plotar_vizinhos(modelo, palavra_teste, topn=20, titulo=f"{nome}: palavras próximas de {palavra_teste}")